# CISC7204 Class E Team 01 - Proposal figures (colour version)
# Data: data_processed/ of the team repository (CISC7204-Team01).
# Figure files are written to figures/ of the same repository.
# Core analysis window: 2023-01 .. 2025-12 (36 months, pandemic years excluded).
# Long-run trend (context only): 2015-01 .. 2026-09 with the pandemic years drawn as a gap.
# Captions live in the report, so the in-figure titles stay short and do not repeat them.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap

REPO = r"C:\Users\07viv\Desktop\CISC7204-Team01"
D = os.path.join(REPO, "data_processed")
OUT = os.path.join(REPO, "figures")
os.makedirs(OUT, exist_ok=True)

# Okabe-Ito colourblind-safe palette
C = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00",
         purple="#CC79A7", sky="#56B4E9", yellow="#F0E442", grey="#8C8C8C")

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.titlesize": 10.5, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.edgecolor": "#444444",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#DDDDDD", "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.dpi": 200, "savefig.bbox": "tight",
})

WIN_START, WIN_END = "2023-01-01", "2025-12-01"

In [2]:
monthly = pd.read_csv(os.path.join(D, "monthly_panel_2023_01_2026_09.csv"))
arr = pd.read_csv(os.path.join(D, "demand_monthly_arrivals.csv"))
arr["date"] = pd.to_datetime(arr["date"])
quarterly = pd.read_csv(os.path.join(D, "quarterly_panel_2023Q1_2026Q3.csv"))
zone = pd.read_csv(os.path.join(D, "match_zone_supply_demand.csv"))
hourly = pd.read_csv(os.path.join(D, "match_hourly_profile.csv"))
panel = pd.read_csv(os.path.join(D, "demand_district_hour_panel_2023_2026.csv"))

win = monthly[(monthly.month >= "2023-01") & (monthly.month <= "2025-12")].copy()
qwin = quarterly[quarterly.quarter <= "2025Q4"].copy()
r_month = win.visitor_arrivals.corr(win.bus_monthly_trips)
r_quarter = qwin.visitor_arrivals_sum.corr(qwin.bus_monthly_trips_sum)
r_trend = np.corrcoef(np.arange(len(qwin)), qwin.bus_trips_per_arrival)[0, 1]  # panel (b) own coefficient, against time
print("monthly r (n=36) =", round(r_month, 3), "| quarterly r (n=12) =", round(r_quarter, 3))

monthly r (n=36) = 0.759 | quarterly r (n=12) = 0.917


# ## Figure 1 - long-run trend with the 36-month core window marked

In [3]:
fig, ax = plt.subplots(figsize=(9.4, 3.6))
s = arr[(arr.date >= "2015-01-01") & (arr.date <= "2026-09-30")].sort_values("date")
pand = s[(s.date >= "2020-01-01") & (s.date <= "2022-12-31")]
rest = s[~s.date.isin(pand.date)]
ax.axvspan(pd.Timestamp("2020-01-01"), pd.Timestamp("2022-12-31"), color=C["grey"], alpha=0.18, lw=0)
ax.plot(rest.date, rest.arrivals / 1e6, color=C["blue"], lw=1.9, label="Arrivals per month")
ax.plot(pand.date, pand.arrivals / 1e6, color=C["red"], lw=1.9, ls=(0, (4, 2)), label="Pandemic years (excluded)")
ax.axvspan(pd.Timestamp(WIN_START), pd.Timestamp("2025-12-31"), color=C["orange"], alpha=0.16, lw=0)
ax.annotate("Core analysis window\n2023-01 to 2025-12 (36 months)",
            xy=(pd.Timestamp("2025-12-01"), 3.0), xytext=(pd.Timestamp("2021-11-01"), 4.62),
            fontsize=8.5, color="#8a4b00", ha="left",
            arrowprops=dict(arrowstyle="->", color="#8a4b00", lw=1.2))
ax.set_ylabel("Arrivals (millions per month)")
ax.set_title("Arrivals per month, 2015-2026")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_ylim(0, 5.2)
ax.legend(loc="lower right", ncol=1, fontsize=8)
fig.savefig(os.path.join(OUT, "fig1_general_trend.png"))
plt.close(fig)

# ## Figure 2 - zone level supply-demand mismatch

In [4]:
fig, ax = plt.subplots(figsize=(7.2, 3.2))
x = np.arange(len(zone)); w = 0.26
ax.bar(x - w, zone.demand_share_pct, w, color=C["blue"], label="Visitor demand share")
ax.bar(x, zone.departures_share_pct, w, color=C["orange"], label="Bus departure share")
ax.bar(x + w, zone.stops_share_pct, w, color=C["purple"], label="Bus stop share")
for i, (d, b) in enumerate(zip(zone.demand_share_pct, zone.departures_share_pct)):
    ax.text(i - w, d + 1.2, f"{d:.1f}%", ha="center", fontsize=8, color="#00507d")
    ax.text(i, b + 1.2, f"{b:.1f}%", ha="center", fontsize=8, color="#8a4b00")
ax.set_xticks(x); ax.set_xticklabels(zone.zone)
ax.set_ylabel("Share of city total (%)"); ax.set_ylim(0, 78)
ax.set_title("Where visitors are, and where the buses are")
ax.legend(loc="upper right", fontsize=8)
fig.savefig(os.path.join(OUT, "fig2_zone_mismatch.png"))
plt.close(fig)

# ## Figure 3 - hourly profile of demand versus supply

In [5]:
fig, ax = plt.subplots(figsize=(7.6, 3.3))
ax.plot(hourly.hour, hourly.demand_share_pct, color=C["red"], lw=2.2, marker="o", ms=3.4,
        label="Visitor demand (hourly share)")
ax.plot(hourly.hour, hourly.supply_share_weekday_pct, color=C["green"], lw=1.9, marker="s", ms=3.0,
        label="Bus departures, weekday")
ax.plot(hourly.hour, hourly.supply_share_weekend_pct, color=C["sky"], lw=1.9, ls="--", marker="^", ms=3.0,
        label="Bus departures, weekend")
ax.fill_between(hourly.hour, hourly.supply_share_weekday_pct, hourly.demand_share_pct,
                where=hourly.demand_share_pct > hourly.supply_share_weekday_pct,
                color=C["red"], alpha=0.12, lw=0, label="Undersupplied hours")
ax.axvspan(21.5, 23.5, color=C["purple"], alpha=0.13, lw=0)
ax.annotate("Evening demand\nwith almost no service", xy=(22.4, 3.4), xytext=(15.4, 6.3),
            fontsize=8.2, color="#7a2f63",
            arrowprops=dict(arrowstyle="->", color="#7a2f63", lw=1.1))
ax.set_xlabel("Hour of day"); ax.set_ylabel("Share of daily total (%)")
ax.set_xticks(range(0, 24, 2)); ax.set_xlim(-0.5, 23.5)
ax.set_title("Hourly demand and supply profiles")
ax.legend(loc="upper left", fontsize=7.8, ncol=2)
fig.savefig(os.path.join(OUT, "fig3_hourly_profile.png"))
plt.close(fig)

# ## Figure 4 - correlation inside the 36-month window, and the quarterly view

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.2))
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.05, 0.85, 3))
for c, yr in zip(colors, [2023, 2024, 2025]):
    sub = win[win.year == yr]
    ax.scatter(sub.visitor_arrivals / 1e6, sub.bus_monthly_trips / 1e6, s=34, color=c,
               edgecolor="white", lw=0.7, label=str(yr), zorder=3)
k, b = np.polyfit(win.visitor_arrivals / 1e6, win.bus_monthly_trips / 1e6, 1)
xs = np.linspace(win.visitor_arrivals.min() / 1e6, win.visitor_arrivals.max() / 1e6, 20)
ax.plot(xs, k * xs + b, color=C["red"], lw=2.0, zorder=2)
ax.set_xlabel("Visitor arrivals (millions per month)")
ax.set_ylabel("Bus passenger trips (millions per month)")
ax.set_title("(a) Monthly, n = 36", fontsize=9.5)
ax.text(0.04, 0.94, f"r = {r_month:.3f}", transform=ax.transAxes, fontsize=9.5,
        color=C["red"], weight="bold", va="top",
        bbox=dict(boxstyle="round,pad=0.22", fc="white", ec="none", alpha=0.85))
ax.legend(loc="lower right", fontsize=8, title=None)

ax = axes[1]
q = qwin.reset_index(drop=True)
ax.bar(q.quarter, q.bus_trips_per_arrival, color=C["green"], width=0.62)
for i, (qq, v) in enumerate(zip(q.quarter, q.bus_trips_per_arrival)):
    ax.text(i, v + 0.2, f"{v:.1f}", ha="center", fontsize=7.4, color="#00543d")
ax.set_ylabel("Bus trips per visitor")
ax.set_ylim(0, 14.2)
ax.set_title("(b) Quarterly, n = 12 quarters", fontsize=9.5)
ax.tick_params(axis="x", rotation=45, labelsize=7.2)
# panel (b) prints its OWN coefficient (against time); the quarterly coefficient
# between arrivals and trips (r_quarter) belongs in the text, not on this panel.
ax.text(0.03, 0.97, f"r = {r_trend:.3f} against time", transform=ax.transAxes, fontsize=8.8,
        color="#7a2f00", weight="bold", va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#E69F00", lw=0.7, alpha=0.95))
ax.annotate("ridership per visitor\nfalls by roughly 40%", xy=(11.0, 5.6), xytext=(6.1, 11.3),
            fontsize=8.0, color="#7a2f00", ha="left",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.9),
            arrowprops=dict(arrowstyle="->", color="#7a2f00", lw=1.1))
fig.tight_layout()
fig.savefig(os.path.join(OUT, "fig4_correlation.png"))
plt.close(fig)


# ## Figure 5 - district by hour demand heatmap

In [7]:
ym = panel.year * 100 + panel.month
# strictly 2023-01 .. 2025-12: the 2026 months are context only and must not enter Figure 5
p36 = panel[(ym >= 202301) & (ym <= 202512)]
top = p36.groupby("district_en").headcount.sum().sort_values(ascending=False).head(10).index.tolist()
hm = (p36[p36.district_en.isin(top)]
      .groupby(["district_en", "hour"]).headcount.sum().unstack("hour"))
hm = hm.reindex(top)
hm = hm.div(hm.sum(axis=1), axis=0) * 100

import textwrap
fig, ax = plt.subplots(figsize=(9.0, 4.4))
cmap = LinearSegmentedColormap.from_list("dem", ["#FFFFFF", "#FFE08A", "#F4A259", "#E4572E", "#8C1C13"])
im = ax.imshow(hm.values, aspect="auto", cmap=cmap)
ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2))
ax.set_yticks(range(len(hm.index)))
ax.set_yticklabels([textwrap.fill(str(v), 26) for v in hm.index], fontsize=7.6)
ax.set_xlabel("Hour of day")
ax.grid(False)
cb = fig.colorbar(im, ax=ax, pad=0.015)
cb.set_label("Share of district daily demand (%)", fontsize=8)
ax.set_title("Ten busiest districts by hour, 2023-2025")
fig.savefig(os.path.join(OUT, "fig5_district_hour_heatmap.png"))
plt.close(fig)

# ## Figure 6 - twelve month work plan, with the owner of each work package
# The right-hand column names who owns each work package; the owners follow the module allocation
# (M1 Lucas, M2 Ray, M3 Vivi) and the roles in the collaboration table.

In [8]:
# Figure 6 - delivery plan on the four-week timeline (W1-W4, 5 Nov - 3 Dec 2026).
# Labels sit inside the bars, the owners are named per week in the plan of Section 7, and the
# critical chain is
# marked with a red outline instead of an arrow, so nothing overlaps at print size.
tasks = [
    # (work package (short label), start week index (0 = W1), duration in weeks, colour, critical)
    ("Audit, cleaning,\ndictionaries (M1-M3)", 0, 1, C["blue"],   True),
    ("Master panel merge\nand first EDA",      1, 1, C["sky"],    False),
    ("Long-run context\nchart (Figure 1)",     1, 1, C["green"],  False),
    ("Zone and hour\nmatching layers",         1, 2, C["orange"], True),
    ("Correlation and\nseasonality",           2, 1, C["purple"], False),
    ("Dashboard\nprototype",                   2, 1, C["red"],    True),
    ("Report draft,\nQC and rehearsal",        3, 1, C["yellow"], True),
]

fig, ax = plt.subplots(figsize=(7.1, 2.85))
for i, (name, start, dur, col, crit) in enumerate(tasks):
    ax.barh(i, dur, left=start, height=0.56, color=col,
            edgecolor=C["red"] if crit else "white",
            linewidth=1.9 if crit else 1.0, zorder=3)
    ax.text(start + dur / 2, i, f"{dur}w", ha="center", va="center",
            fontsize=7.6, weight="bold", color="white", zorder=4)

ax.set_yticks(range(len(tasks)))
ax.set_yticklabels([t[0] for t in tasks], fontsize=8.2, linespacing=1.15)
ax.set_xlim(0, 4)
ax.set_ylim(len(tasks) - 0.5, -0.75)
ax.set_xticks([0.5, 1.5, 2.5, 3.5])
ax.set_xticklabels(["W1\n5-11 Nov", "W2\n12-18 Nov", "W3\n19-25 Nov", "W4\n26 Nov-3 Dec"], fontsize=7.9)
ax.tick_params(axis="both", length=2.6, colors="#444444")
ax.grid(axis="y", visible=False)
ax.set_axisbelow(True)
for x in (1, 2, 3):
    ax.axvline(x, color="#BFBFBF", lw=0.7, ls=(0, (2, 2)), zorder=1)
ax.set_title("Delivery plan on the four-week timeline", fontsize=8.8, weight="bold", pad=17, loc="left")
ax.legend(handles=[Patch(facecolor="#B0B0B0", edgecolor="white", label="work package"),
                   Patch(facecolor="white", edgecolor=C["red"], linewidth=1.9,
                         label="critical chain (cannot slip)")],
          loc="lower left", bbox_to_anchor=(0.0, 0.995), ncol=2, fontsize=7.6,
          handlelength=1.6, columnspacing=1.4, borderpad=0.1)
fig.text(0.005, -0.05,
         "Names: Lucas = KONG KIN MENG, Ray = KUAN CHI KIT, Vivi = FONG CHIN WAI, "
         "Sonia = FONG HIO LAM, Qilan = XIAN QILAN, Song = SONG ZHUOXI. "
         "Owners per work package are listed in the week-by-week plan of Section 7.",
         ha="left", va="top", fontsize=6.4, color="#555555")
fig.savefig(os.path.join(OUT, "fig6_workplan.png"))
plt.close(fig)

print("figures written:", sorted(os.listdir(OUT)))
print("win rows", len(win), "qwin", len(qwin))
print("trips per visitor first/last:", qwin.bus_trips_per_arrival.iloc[0], qwin.bus_trips_per_arrival.iloc[-1])


figures written: ['archive', 'fig1_general_trend.png', 'fig2_zone_mismatch.png', 'fig3_hourly_profile.png', 'fig4_correlation.png', 'fig5_district_hour_heatmap.png', 'fig6_workplan.png']
win rows 36 qwin 12
trips per visitor first/last: 9.9637 5.8254
